# Step 3 ? Robustness, subgroup analysis, and confidence intervals

Run this notebook **after** `STEP_2_CALIBRATION_AND_ALERTS.ipynb`. It uses the untouched final-test predictions generated there to report:

- warning performance by state, broad region, season, and high-risk city group;
- date-block bootstrap 95% confidence intervals for recall, precision, and false-positive rate; and
- explicit interpretation guardrails for the manuscript.

The bootstrap samples whole forecast dates, retaining the correlation among cities observed on the same day.


## 1. Imports and inputs

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_score, recall_score

pd.set_option("display.max_columns", 100)

PROJECT_ROOT = Path(r"C:\Users\cdipi\Downloads\aqi_for india")
INPUT = PROJECT_ROOT / "reports" / "calibration" / "calibrated_alert_test_predictions.csv"
OUTPUT_DIR = PROJECT_ROOT / "reports" / "robustness"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Change to 1,000 for the final paper run; 500 is a practical default.
N_BOOTSTRAP = 500
RANDOM_STATE = 42
MIN_GROUP_ROWS = 100
MIN_GROUP_EVENTS = 10

assert INPUT.exists(), (
    f"Run STEP_2_CALIBRATION_AND_ALERTS.ipynb first. Missing: {INPUT}"
)
predictions = pd.read_csv(INPUT, parse_dates=["date", "forecast_date"])
print(f"Loaded {len(predictions):,} policy-level test predictions")
print(f"Tasks: {sorted(predictions.task.unique())}")
print(f"Policies: {sorted(predictions.policy.unique())}")


## 2. Add interpretable geographic and seasonal groups

The original state labels are retained. Broad regions are descriptive reporting groups, not model features. Groups with too few rows or events are marked as insufficient for stable estimates.


In [ ]:
NORTH = {
    "Chandigarh", "Delhi", "Haryana", "Himachal Pradesh", "Jammu and Kashmir", "Ladakh",
    "Punjab", "Rajasthan", "Uttar Pradesh", "Uttarakhand",
}
EAST = {"Bihar", "Jharkhand", "Odisha", "West Bengal"}
NORTHEAST = {
    "Arunachal Pradesh", "Assam", "Manipur", "Meghalaya", "Mizoram", "Nagaland",
    "Sikkim", "Tripura",
}
WEST = {"Goa", "Gujarat", "Maharashtra", "Dadra and Nagar Haveli and Daman and Diu"}
CENTRAL = {"Chhattisgarh", "Madhya Pradesh"}
SOUTH = {"Andhra Pradesh", "Karnataka", "Kerala", "Puducherry", "Tamil Nadu", "Telangana", "Lakshadweep"}

def broad_region(state: str) -> str:
    if state in NORTH: return "North / Indo-Gangetic"
    if state in EAST: return "East"
    if state in NORTHEAST: return "North-East"
    if state in WEST: return "West"
    if state in CENTRAL: return "Central"
    if state in SOUTH: return "South"
    return "Other / unclassified"

def season(month: int) -> str:
    if month in (12, 1, 2): return "Winter"
    if month in (3, 4, 5): return "Summer"
    if month in (6, 7, 8, 9): return "Monsoon"
    return "Post-monsoon"

HIGH_RISK_AREAS = {"Delhi", "Byrnihat", "Greater Noida", "Noida", "Chhapra", "Bhagalpur", "Begusarai", "Saharsa"}
predictions["region"] = predictions.state.map(broad_region)
predictions["season"] = predictions.forecast_date.dt.month.map(season)
predictions["high_risk_group"] = np.where(predictions.area.isin(HIGH_RISK_AREAS), "High-risk locations", "Other locations")


## 3. Subgroup metrics

Recall and precision are computed after the policy threshold has been fixed. This quantifies where the model may be more or less reliable. Do not rank groups with fewer than 10 observed events.


In [ ]:
def group_metrics(group: pd.DataFrame) -> pd.Series:
    y_true = group.event_observed.to_numpy(dtype=int)
    y_pred = group.alert_issued.to_numpy(dtype=int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    events = int(y_true.sum())
    rows = len(group)
    adequate = rows >= MIN_GROUP_ROWS and events >= MIN_GROUP_EVENTS
    return pd.Series({
        "rows": rows, "events": events, "event_rate": events / rows if rows else np.nan,
        "Recall": recall_score(y_true, y_pred, zero_division=0) if adequate else np.nan,
        "Precision": precision_score(y_true, y_pred, zero_division=0) if adequate else np.nan,
        "false_positive_rate": fp / (fp + tn) if adequate and fp + tn else np.nan,
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "sufficient_events": adequate,
    })

subgroup_tables = []
for group_name in ["state", "region", "season", "high_risk_group"]:
    summary = (
        predictions.groupby(["task", "policy", group_name], observed=True)
        .apply(group_metrics, include_groups=False)
        .reset_index()
        .rename(columns={group_name: "group"})
    )
    summary["group_type"] = group_name
    subgroup_tables.append(summary)

subgroups = pd.concat(subgroup_tables, ignore_index=True)
subgroups.to_csv(OUTPUT_DIR / "subgroup_metrics.csv", index=False)
display(subgroups.sort_values(["group_type", "task", "policy", "events"], ascending=[True, True, True, False]).head(40).round(3))

# State chart focuses on safety-first policy, where missed events are the principal concern.
safety = subgroups[(subgroups.policy == "Safety-first (recall target)") & (subgroups.group_type == "state") & subgroups.sufficient_events]
if not safety.empty:
    plot = safety.sort_values("Recall")
    plt.figure(figsize=(10, max(5, len(plot) * 0.28)))
    sns.barplot(data=plot, y="group", x="Recall", hue="task", orient="h")
    plt.xlim(0, 1); plt.xlabel("Final-test recall"); plt.ylabel("State")
    plt.title("State-level event recall (sufficient-event groups only)")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "state_recall_safety_first.png", dpi=160)
    plt.show()


## 4. Date-block bootstrap confidence intervals

Each bootstrap replicate samples forecast dates with replacement, then keeps every city record from selected dates. This avoids treating same-day city observations as independent. The interval is percentile-based and should be reported beside the point metric.


In [ ]:
def daily_confusion_counts(group: pd.DataFrame) -> pd.DataFrame:
    daily = []
    for date, one_day in group.groupby("forecast_date", observed=True):
        y_true = one_day.event_observed.to_numpy(dtype=int)
        y_pred = one_day.alert_issued.to_numpy(dtype=int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        daily.append((date, tp, fp, fn, tn))
    return pd.DataFrame(daily, columns=["forecast_date", "TP", "FP", "FN", "TN"])

def bootstrap_intervals(group: pd.DataFrame, n_bootstrap: int, seed: int) -> dict:
    daily = daily_confusion_counts(group)
    values = daily[["TP", "FP", "FN", "TN"]].to_numpy(dtype=float)
    rng = np.random.default_rng(seed)
    results = []
    for _ in range(n_bootstrap):
        sample = values[rng.integers(0, len(values), size=len(values))].sum(axis=0)
        tp, fp, fn, tn = sample
        recall = tp / (tp + fn) if tp + fn else np.nan
        precision = tp / (tp + fp) if tp + fp else np.nan
        fpr = fp / (fp + tn) if fp + tn else np.nan
        results.append((recall, precision, fpr))
    values = np.asarray(results, dtype=float)
    point = daily[["TP", "FP", "FN", "TN"]].sum().to_numpy(dtype=float)
    tp, fp, fn, tn = point
    return {
        "forecast_dates": len(daily),
        "Recall": tp / (tp + fn),
        "Recall_ci_low": np.nanpercentile(values[:, 0], 2.5),
        "Recall_ci_high": np.nanpercentile(values[:, 0], 97.5),
        "Precision": tp / (tp + fp),
        "Precision_ci_low": np.nanpercentile(values[:, 1], 2.5),
        "Precision_ci_high": np.nanpercentile(values[:, 1], 97.5),
        "false_positive_rate": fp / (fp + tn),
        "false_positive_rate_ci_low": np.nanpercentile(values[:, 2], 2.5),
        "false_positive_rate_ci_high": np.nanpercentile(values[:, 2], 97.5),
    }

bootstrap_rows = []
for (task, policy), group in predictions.groupby(["task", "policy"], observed=True):
    bootstrap_rows.append({
        "task": task, "policy": policy,
        **bootstrap_intervals(group, N_BOOTSTRAP, RANDOM_STATE),
    })

bootstrap_table = pd.DataFrame(bootstrap_rows).sort_values(["task", "policy"])
bootstrap_table.to_csv(OUTPUT_DIR / "date_block_bootstrap_confidence_intervals.csv", index=False)
display(bootstrap_table.round(3))


## 5. Manuscript-ready interpretation

Use the files generated here to report subgroup variability and 95% confidence intervals. The final paper must retain these statements:

- NASA POWER data are historical reanalysis, not operational weather forecasts.
- The evaluation predicts later dates for locations already observed in training; it does not establish performance for entirely unseen cities.
- Severe-event alerts should report precision and false-alert burden alongside recall.
- Small state/city strata must be labelled insufficient rather than ranked.


In [ ]:
manifest = {
    "input": str(INPUT.relative_to(PROJECT_ROOT)),
    "bootstrap_method": "forecast-date block bootstrap",
    "bootstrap_replicates": N_BOOTSTRAP,
    "confidence_interval": "2.5th to 97.5th percentile",
    "minimum_group_rows": MIN_GROUP_ROWS,
    "minimum_group_events": MIN_GROUP_EVENTS,
}
with (OUTPUT_DIR / "robustness_manifest.json").open("w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)
print("Created:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path.relative_to(PROJECT_ROOT))
